In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import torch

from tsfm_lens.utils import (
    apply_custom_style,
)

In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
# read in pickle file
model_name = "chronos-t5-base"
config_key = "rf4_sl32"

induction_results_dir = "../../outputs/rrt_induction_scores"
with open(os.path.join(induction_results_dir, model_name, config_key, "center_scores.pkl"), "rb") as f:
    center_scores = pickle.load(f)
    all_center_scores = center_scores["all_center_scores"]
    center_scores_mean, center_scores_std = center_scores["mean"], center_scores["std"]

with open(os.path.join(induction_results_dir, model_name, config_key, "right_scores.pkl"), "rb") as f:
    right_scores = pickle.load(f)
    all_right_scores = right_scores["all_right_scores"]
    right_scores_mean, right_scores_std = right_scores["mean"], right_scores["std"]

with open(
    os.path.join(induction_results_dir, model_name, config_key, "correct_token_attribution.pkl"),
    "rb",
) as f:
    correct_token_attribution = pickle.load(f)
    all_correct_token_attributions = correct_token_attribution["correct_token_attributions"]
    all_token_attributions = correct_token_attribution["all_token_attributions"]
    correct_token_attribution_mean, correct_token_attribution_std = (
        correct_token_attribution["mean"],
        correct_token_attribution["std"],
    )

with open(os.path.join(induction_results_dir, model_name, config_key, "rrt_vars.pkl"), "rb") as f:
    rrt_vars = pickle.load(f)
    std_factor, overall_scores_mean, overall_scores_std = (
        rrt_vars["std_factor"],
        rrt_vars["overall_scores"][0],
        rrt_vars["overall_scores"][1],
    )

entropy_results_dir = "../../outputs/head_entropy"
with open(os.path.join(entropy_results_dir, "head_entropy_ca.pkl"), "rb") as f:
    head_entropy = pickle.load(f)
    avg_ent_l_h = torch.tensor(head_entropy["avg_ent_l_h"])

In [ ]:
print("=" * 80)
print("DATA SHAPES")
print("=" * 80)
print(f"all_center_scores shape: {all_center_scores.shape}")
print("  Description: Center position induction scores")
print("  Dimensions: (batch, layers, heads)\n")

print(f"all_right_scores shape: {all_right_scores.shape}")
print("  Description: Right position induction scores")
print("  Dimensions: (batch, layers, heads)\n")

print(f"all_correct_token_attribution shape: {all_correct_token_attributions.shape}")
print("  Description: Attribution scores for correct tokens")
print("  Dimensions: (batch, layers, heads)\n")

print(f"all_token_attribution shape: {all_token_attributions.shape}")
print("  Description: Attribution scores for all tokens")
print("  Dimensions: (batch, layers, heads, tokens)\n")

print(f"avg_ent_l_h shape: {avg_ent_l_h.shape}")
print("  Description: Average head entropy")
print("  Dimensions: (layers, heads)\n")

# print("=" * 80)
# print("SUMMARY STATISTICS")
# print("=" * 80)
# print(f"Model: {model_name}")
# print(f"Config: {config_key}")
# print(f"Std Factor: {std_factor}\n")

# print(f"Overall Scores:")
# print(f"  Mean: {overall_scores_mean:.4f}")
# print(f"  Std:  {overall_scores_std:.4f}\n")

# print(f"Center Scores: {center_scores_mean.shape} (layers x heads)")
# print(f"  Mean range: [{center_scores_mean.min():.4f}, {center_scores_mean.max():.4f}]")
# print(f"  Std range:  [{center_scores_std.min():.4f}, {center_scores_std.max():.4f}]\n")

# print(f"Right Scores: {right_scores_mean.shape} (layers x heads)")
# print(f"  Mean range: [{right_scores_mean.min():.4f}, {right_scores_mean.max():.4f}]")
# print(f"  Std range:  [{right_scores_std.min():.4f}, {right_scores_std.max():.4f}]\n")

# print(f"Correct Token Attribution: {correct_token_attribution_mean.shape} (layers x heads)")
# print(f"  Mean range: [{correct_token_attribution_mean.min():.4f}, {correct_token_attribution_mean.max():.4f}]")
# print(f"  Std range:  [{correct_token_attribution_std.min():.4f}, {correct_token_attribution_std.max():.4f}]")

In [ ]:
center_scores_mean = all_center_scores.mean(dim=0)
right_scores_mean = all_right_scores.mean(dim=0)

# set induction_scores_mean to be the element-wise maximum of center_scores_mean and right_scores_mean
induction_scores_mean = torch.max(center_scores_mean, right_scores_mean)
correct_token_attribution_mean = all_correct_token_attributions.mean(dim=0)

In [ ]:
relative_attribution_scores = (
    all_correct_token_attributions - all_token_attributions.mean(dim=-1)
) / all_token_attributions.std(dim=-1)
relative_attributions_mean = relative_attribution_scores.mean(dim=0)
relative_attributions_mean.shape

In [ ]:
# plot heatmap of average prefix scores with y axis being layer and x axis being head index

scores = right_scores_mean.detach().cpu().numpy()  # [layers, heads]

# robust color scaling (avoids a single large head dominating the colormap)
vmin = float(np.quantile(scores, 0.01))
vmax = float(np.quantile(scores, 0.99))

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(scores, aspect="equal", origin="lower", cmap="Blues", vmin=0, vmax=1)

ax.set_title("Average prefix score", fontweight="bold")
ax.set_xlabel("Head index", fontweight="bold")
ax.set_ylabel("Layer", fontweight="bold")

n_layers, n_heads = scores.shape
ax.set_xticks(np.arange(n_heads))
ax.set_xticklabels(np.arange(1, n_heads + 1))
ax.set_yticks(np.arange(n_layers))
ax.set_yticklabels(np.arange(1, n_layers + 1))

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Average prefix score", fontweight="bold")

fig.tight_layout()

# save as prefix_mosaic.svg/pdf
out_dir = "../../figs/induction_heads/mosaic"
os.makedirs(out_dir, exist_ok=True)
fig.savefig(os.path.join(out_dir, "prefix_mosaic.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(out_dir, "prefix_mosaic.svg"), bbox_inches="tight")

plt.show()

# --- Separate heatmap: average head entropy (normalized to [0, 1] for plotting only) ---
ent = avg_ent_l_h.detach().cpu().numpy()  # [layers, heads]
ent_min = float(ent.min())
ent_max = float(ent.max())
denom = ent_max - ent_min
if denom == 0.0:
    ent_norm = np.zeros_like(ent, dtype=np.float64)
else:
    ent_norm = (ent - ent_min) / denom

fig2, ax2 = plt.subplots(figsize=(7, 7))
im2 = ax2.imshow(ent_norm, aspect="equal", origin="lower", cmap="Blues_r", vmin=0.0, vmax=1.0)

ax2.set_title("Average head entropy (normalized)", fontweight="bold")
ax2.set_xlabel("Head index", fontweight="bold")
ax2.set_ylabel("Layer", fontweight="bold")

n_layers_e, n_heads_e = ent_norm.shape
ax2.set_xticks(np.arange(n_heads_e))
ax2.set_xticklabels(np.arange(1, n_heads_e + 1))
ax2.set_yticks(np.arange(n_layers_e))
ax2.set_yticklabels(np.arange(1, n_layers_e + 1))

cbar2 = fig2.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
cbar2.set_label("Entropy (normalized)", fontweight="bold")

fig2.tight_layout()

# optional save alongside prefix mosaic
fig2.savefig(os.path.join(out_dir, "entropy_mosaic_norm.pdf"), bbox_inches="tight")
fig2.savefig(os.path.join(out_dir, "entropy_mosaic_norm.svg"), bbox_inches="tight")

plt.show()

In [ ]:
correct_token_threshold = 2.0
right_score_threshold = 0.3
entropy_threshold = 4.0

In [ ]:
# 3x3 lower-triangular grid for:
#   (1) prefix score
#   (2) copy score
#   (3) entropy (normalized)


def _to_1d_numpy(a):
    if isinstance(a, torch.Tensor):
        return a.detach().cpu().flatten().numpy()
    return np.asarray(a).flatten()


def _minmax_norm_1d(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x_min = float(np.min(x))
    x_max = float(np.max(x))
    denom = x_max - x_min
    if denom == 0.0:
        return np.zeros_like(x, dtype=np.float64)
    return (x - x_min) / denom


# Config: normalize entropy axis to [0, 1] for display
normalize_entropy_for_plot = True

x_prefix = _to_1d_numpy(right_scores_mean)
y_copy = _to_1d_numpy(relative_attributions_mean)
z_ent_raw = _to_1d_numpy(avg_ent_l_h)  # [layers, heads] -> flat
z_ent = _minmax_norm_1d(z_ent_raw) if normalize_entropy_for_plot else z_ent_raw

vars_ = [x_prefix, y_copy, z_ent]
labels = [
    "Avg Prefix Score",
    "Avg Copy Score",
    "Avg Entropy \n (normalized)" if normalize_entropy_for_plot else "Avg Entropy",
]

fig, axes = plt.subplots(3, 3, figsize=(7, 7), sharex="col", sharey=False)

for i in range(3):
    for j in range(3):
        ax = axes[i, j]

        if i == j:
            ax.hist(vars_[i], bins=30, alpha=0.8, histtype="stepfilled", color="tab:blue")
            ax.set_title(labels[i], fontweight="bold")
            ax.set_ylabel("Count")
            ax.grid(True, alpha=0.3)

        elif i > j:
            ax.scatter(vars_[j], vars_[i], alpha=0.6, s=18, color="tab:blue")
            ax.grid(True, alpha=0.3)

        else:
            ax.axis("off")

        if i == 2 and i > j:
            ax.set_xlabel(labels[j])
        if j == 0 and i > j:
            ax.set_ylabel(labels[i])

plt.tight_layout()
plt.show()

# save figure as pdf and svg
out_dir = "../../figs/induction_heads/triangle_plot"
os.makedirs(out_dir, exist_ok=True)
fig.savefig(os.path.join(out_dir, "prefix_attribution_entropy.pdf"))
fig.savefig(os.path.join(out_dir, "prefix_attribution_entropy.svg"))

In [ ]:
# Proportions at the chosen thresholds

num_heads = relative_attributions_mean.numel()

num_heads_above_threshold = (relative_attributions_mean > correct_token_threshold).sum().item()
print("P(copy score > threshold):\n", num_heads_above_threshold / num_heads)

num_heads_prefix_above_threshold = (right_scores_mean > right_score_threshold).sum().item()
print("P(prefix score > threshold):\n", num_heads_prefix_above_threshold / num_heads)

# Normalize entropy for reporting only (does not mutate stored tensors)
ent_min = float(avg_ent_l_h.detach().cpu().min().item())
ent_max = float(avg_ent_l_h.detach().cpu().max().item())
ent_denom = ent_max - ent_min if ent_max != ent_min else 1.0
avg_ent_norm = (avg_ent_l_h.detach().cpu() - ent_min) / ent_denom
normalized_entropy_threshold = float((float(entropy_threshold) - ent_min) / ent_denom)

num_heads_entropy_below_threshold = (avg_ent_norm < normalized_entropy_threshold).sum().item()
print("P(entropy (normalized) < threshold):\n", num_heads_entropy_below_threshold / num_heads)

num_heads_both_above_threshold = (
    ((relative_attributions_mean > correct_token_threshold) & (right_scores_mean > right_score_threshold)).sum().item()
)
print("P(induction head):\n", num_heads_both_above_threshold / num_heads)

print("=" * 40)

num_heads_both_above_threshold = (
    ((relative_attributions_mean > correct_token_threshold) & (right_scores_mean > right_score_threshold)).sum().item()
)
num_heads_prefix_condition = (right_scores_mean > right_score_threshold).sum().item()
print(
    "P(copy score > threshold | prefix score > threshold):\n",
    num_heads_both_above_threshold / num_heads_prefix_condition,
)

num_heads_entropy_below_threshold = (
    (
        (avg_ent_norm < normalized_entropy_threshold)
        & (relative_attributions_mean > correct_token_threshold)
        & (right_scores_mean > right_score_threshold)
    )
    .sum()
    .item()
)
print(
    "P(entropy (normalized) < threshold | induction head):\n",
    num_heads_entropy_below_threshold / num_heads_both_above_threshold,
)

num_heads_both_above_threshold = (
    (
        (relative_attributions_mean > correct_token_threshold)
        & (right_scores_mean > right_score_threshold)
        & (avg_ent_norm < normalized_entropy_threshold)
    )
    .sum()
    .item()
)
num_heads_entropy_condition = (avg_ent_norm < normalized_entropy_threshold).sum().item()
print(
    "P(induction head | entropy (normalized) < threshold):\n",
    num_heads_both_above_threshold / num_heads_entropy_condition,
)

In [ ]:
# Heatmaps over (copy score threshold, entropy threshold (normalized)) at fixed prefix score threshold
## A "sharp head" means: (entropy < T_e).

# =====================
# Config (edit these)
# =====================
prefix_score_threshold = 0.3
n_Tc = 100
n_Te = 100

# If True, pick plot ranges from data quantiles; if False, use manual ranges below.
use_quantile_bounds = False
Tc_quantiles = (0.01, 0.99)
Te_quantiles = (0.01, 0.99)  # quantiles over avg_ent (raw units)

# Manual ranges (raw units). Note entropy is normalized to [0,1] only for plotting.
Tc_manual = (1.0, 4.0)
Te_manual_raw = (3.0, 8.0)

# When using manual ranges, clamp ONLY the upper entropy bound to the data max.
# This keeps the entropy threshold (normalized) axis within [0, 1] even if you
# accidentally set Te_manual_raw[1] above the observed entropy maximum.
clamp_entropy_upper_to_data_max = True

# =====================
# Data prep
# =====================
rel_attr = relative_attributions_mean.detach().cpu()
right_scores = right_scores_mean.detach().cpu()
avg_ent = avg_ent_l_h.detach().cpu()

# Normalization constants for entropy axis
ent_vals = torch.unique(avg_ent.flatten()).sort().values
ent_min = float(ent_vals[0].item())
ent_max = float(ent_vals[-1].item())
ent_denom = ent_max - ent_min if ent_max != ent_min else 1.0


def _to_ent_norm(x_raw):
    """Map raw entropy values to [0, 1] for plotting (does not mutate stored tensors)."""
    x = np.asarray(x_raw, dtype=np.float64)
    y = (x - ent_min) / ent_denom
    return float(y) if y.shape == () else y


# =====================
# Range selection
# =====================
rel_flat = rel_attr.flatten().numpy()
ent_flat = avg_ent.flatten().numpy()

if use_quantile_bounds:
    Tc_low, Tc_high = (
        float(np.quantile(rel_flat, Tc_quantiles[0])),
        float(np.quantile(rel_flat, Tc_quantiles[1])),
    )
    Te_low_raw, Te_high_raw = (
        float(np.quantile(ent_flat, Te_quantiles[0])),
        float(np.quantile(ent_flat, Te_quantiles[1])),
    )
else:
    Tc_low, Tc_high = map(float, Tc_manual)
    Te_low_raw, Te_high_raw = map(float, Te_manual_raw)

# Clamp ONLY the upper entropy bound (manual mode)
if (not use_quantile_bounds) and clamp_entropy_upper_to_data_max:
    Te_high_raw = min(float(Te_high_raw), float(ent_max))

# Guard against degenerate ranges
if not (Tc_high > Tc_low):
    Tc_high = float(np.nextafter(Tc_low, np.inf))
if not (Te_high_raw > Te_low_raw):
    Te_high_raw = float(np.nextafter(Te_low_raw, np.inf))

copy_score_thresholds = np.linspace(Tc_low, Tc_high, n_Tc)
entropy_thresholds_raw = np.linspace(Te_low_raw, Te_high_raw, n_Te)  # raw entropy units for computation
entropy_thresholds_norm = _to_ent_norm(entropy_thresholds_raw)  # normalized for plotting axis

# =====================
# Compute probability grids
# =====================
# rows=entropy_thresholds_norm (y), cols=copy_score_thresholds (x)
p_sharp_given_copy_prefix = np.full((len(entropy_thresholds_raw), len(copy_score_thresholds)), np.nan, dtype=np.float64)
p_copy_prefix_given_sharp = np.full((len(entropy_thresholds_raw), len(copy_score_thresholds)), np.nan, dtype=np.float64)

for yi, T_e_raw in enumerate(entropy_thresholds_raw):
    mask_ent = avg_ent < float(T_e_raw)
    ent_count = int(mask_ent.sum().item())

    for xi, copy_score_threshold in enumerate(copy_score_thresholds):
        # Induction head mask (terminology): copy score > threshold AND prefix score > threshold
        mask_copy_prefix = (rel_attr > float(copy_score_threshold)) & (right_scores > float(prefix_score_threshold))
        copy_prefix_count = int(mask_copy_prefix.sum().item())

        both_count = int((mask_ent & mask_copy_prefix).sum().item())

        # P(sharp head | induction head)
        if copy_prefix_count > 0:
            p_sharp_given_copy_prefix[yi, xi] = both_count / copy_prefix_count

        # P(induction head | sharp head)
        if ent_count > 0:
            p_copy_prefix_given_sharp[yi, xi] = both_count / ent_count

In [ ]:
# Combined 1x3 plot for the last three heatmap-style figures:
#   (1) P(sharp head | induction head)     vs (copy threshold, entropy threshold (normalized))
#   (2) P(induction head | sharp head)     vs (copy threshold, entropy threshold (normalized))
#   (3) Proportion of induction heads      vs (copy threshold, prefix threshold)

# Requires these from the previous cell:
#   p_sharp_given_copy_prefix, p_copy_prefix_given_sharp, copy_score_thresholds, entropy_thresholds_norm, _to_ent_norm

# ---------------------
# Contour plot data prep
# ---------------------
rel_attr = relative_attributions_mean.detach().cpu()
right_scores = right_scores_mean.detach().cpu()

rel_attr_flat = rel_attr.flatten()
right_flat = right_scores.flatten()

# =====================
# Config (edit these)
# =====================
n_Tc = 100  # x-resolution (copy threshold)
n_Tr = 100  # y-resolution (prefix threshold)

# If True, pick ranges from data quantiles; if False, use the manual ranges below.
use_quantile_bounds = False
Tc_quantiles = (0.01, 0.99)
Tr_quantiles = (0.01, 0.99)

# Manual ranges (only used when use_quantile_bounds=False)
Tc_manual = (1.0, 5.0)
Tr_manual = (0.25, 0.8)

# Color scale: choose a robust min/max from the computed probability grid
use_robust_color_scale = True
p_color_quantiles = (0.01, 0.99)  # (low, high) quantiles over p_both
p_color_floor = 0.0  # force vmin >= this (often 0.0)
n_color_levels = 21  # number of filled contour levels
add_contour_lines = True

# Marker for the thresholds used earlier in the notebook
show_threshold_marker = True
marker_color = "red"
marker_size = 140
marker_linewidth = 2.5

# =====================
# Range selection (contour plot)
# =====================
rel_np = rel_attr_flat.numpy()
right_np = right_flat.numpy()

if use_quantile_bounds:
    Tc_low, Tc_high = (
        float(np.quantile(rel_np, Tc_quantiles[0])),
        float(np.quantile(rel_np, Tc_quantiles[1])),
    )
    Tr_low, Tr_high = (
        float(np.quantile(right_np, Tr_quantiles[0])),
        float(np.quantile(right_np, Tr_quantiles[1])),
    )
else:
    Tc_low, Tc_high = map(float, Tc_manual)
    Tr_low, Tr_high = map(float, Tr_manual)

# Guard against degenerate ranges
if not (Tc_high > Tc_low):
    Tc_high = float(np.nextafter(Tc_low, np.inf))
if not (Tr_high > Tr_low):
    Tr_high = float(np.nextafter(Tr_low, np.inf))

copy_score_thresholds_contour = np.linspace(Tc_low, Tc_high, n_Tc)  # x-axis
prefix_score_thresholds = np.linspace(Tr_low, Tr_high, n_Tr)  # y-axis

# =====================
# Compute contour grid
# =====================
# Grid: rows=prefix_score_thresholds (y), cols=copy_score_thresholds_contour (x)
p_both = np.zeros((len(prefix_score_thresholds), len(copy_score_thresholds_contour)), dtype=np.float64)
num_heads = float(rel_attr_flat.numel())

for yi, prefix_score_threshold in enumerate(prefix_score_thresholds):
    for xi, copy_score_threshold in enumerate(copy_score_thresholds_contour):
        mask = (rel_attr_flat > float(copy_score_threshold)) & (right_flat > float(prefix_score_threshold))
        p_both[yi, xi] = float(mask.sum().item()) / num_heads

# =====================
# Color scaling (data-driven)
# =====================
p_flat = p_both.ravel()
p_min = float(np.nanmin(p_flat))
p_max = float(np.nanmax(p_flat))

if use_robust_color_scale:
    p_lo = float(np.quantile(p_flat, p_color_quantiles[0]))
    p_hi = float(np.quantile(p_flat, p_color_quantiles[1]))
else:
    p_lo, p_hi = p_min, p_max

p_lo = max(float(p_color_floor), p_lo)
if not (p_hi > p_lo):
    p_hi = float(np.nextafter(p_lo, np.inf))

levels = np.linspace(p_lo, p_hi, n_color_levels)

# =====================
# Common setup for all plots
# =====================
extent_hm = [
    float(copy_score_thresholds.min()),
    float(copy_score_thresholds.max()),
    float(np.min(entropy_thresholds_norm)),
    float(np.max(entropy_thresholds_norm)),
]

cmap_hm = plt.cm.viridis.copy()
cmap_hm.set_bad(color="lightgray")

out_dir = "../../figs/induction_heads/threshold_heatmaps"
os.makedirs(out_dir, exist_ok=True)

# =====================
# Plot 1: P(sharp head | induction head)
# =====================
fig1, ax1 = plt.subplots(figsize=(7, 6))
# ax1.set_box_aspect(1)  # Makes the plot box square
im1 = ax1.imshow(
    np.ma.masked_invalid(p_sharp_given_copy_prefix),
    origin="lower",
    aspect="auto",
    extent=extent_hm,
    vmin=0.0,
    vmax=1.0,
    cmap=cmap_hm,
)
ax1.set_title("P(sharp head | induction head)", fontweight="bold")
ax1.set_xlabel("Copy score threshold", fontweight="bold")
ax1.set_ylabel("Entropy threshold (normalized)", fontweight="bold")
cbar1 = fig1.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
cbar1.set_label("Probability", fontweight="bold")

if show_threshold_marker:
    try:
        hm_x = float(correct_token_threshold)
        hm_y = float(_to_ent_norm(float(entropy_threshold)))
        ax1.scatter([hm_x], [hm_y], marker="x", c=marker_color, s=marker_size, linewidths=marker_linewidth, zorder=10)
    except NameError:
        print("WARNING: thresholds not defined; skipping marker.")

fig1.tight_layout()
fig1.savefig(os.path.join(out_dir, "p_sharp_given_induction.pdf"))
fig1.savefig(os.path.join(out_dir, "p_sharp_given_induction.svg"))
plt.show()

# =====================
# Plot 2: P(induction head | sharp head)
# =====================
fig2, ax2 = plt.subplots(figsize=(7, 6))
# ax2.set_box_aspect(1)  # Makes the plot box square
im2 = ax2.imshow(
    np.ma.masked_invalid(p_copy_prefix_given_sharp),
    origin="lower",
    aspect="auto",
    extent=extent_hm,
    vmin=0.0,
    vmax=1.0,
    cmap=cmap_hm,
)
ax2.set_title("P(induction head | sharp head)", fontweight="bold")
ax2.set_xlabel("Copy score threshold", fontweight="bold")
ax2.set_ylabel("Entropy threshold (normalized)", fontweight="bold")
cbar2 = fig2.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
cbar2.set_label("Probability", fontweight="bold")

if show_threshold_marker:
    try:
        hm_x = float(correct_token_threshold)
        hm_y = float(_to_ent_norm(float(entropy_threshold)))
        ax2.scatter([hm_x], [hm_y], marker="x", c=marker_color, s=marker_size, linewidths=marker_linewidth, zorder=10)
    except NameError:
        print("WARNING: thresholds not defined; skipping marker.")

fig2.tight_layout()
fig2.savefig(os.path.join(out_dir, "p_induction_given_sharp.pdf"))
fig2.savefig(os.path.join(out_dir, "p_induction_given_sharp.svg"))
plt.show()

# =====================
# Plot 3: Proportion of induction heads
# =====================
fig3, ax3 = plt.subplots(figsize=(7, 6))
# ax3.set_box_aspect(1)  # Makes the plot box square (not data units)
cf = ax3.contourf(
    copy_score_thresholds_contour,
    prefix_score_thresholds,
    p_both,
    levels=levels,
    cmap="viridis",
    extend="max",
)
if add_contour_lines:
    ax3.contour(
        copy_score_thresholds_contour,
        prefix_score_thresholds,
        p_both,
        levels=levels,
        colors="k",
        linewidths=0.3,
        alpha=0.35,
    )
ax3.set_title("Proportion of induction heads", fontweight="bold")
ax3.set_xlabel("Copy score threshold", fontweight="bold")
ax3.set_ylabel("Prefix score threshold", fontweight="bold")
cbar3 = fig3.colorbar(cf, ax=ax3, fraction=0.046, pad=0.04, format="%.3f")
cbar3.set_label("Proportion of heads", fontweight="bold")
cbar3.ax.set_title(f"[{p_lo:.3f}, {p_hi:.3f}]", fontsize=9)
fig3.tight_layout()

if show_threshold_marker:
    try:
        ct = float(correct_token_threshold)
        rt = float(right_score_threshold)
        ax3.scatter(
            [ct],
            [rt],
            marker="x",
            c=marker_color,
            s=marker_size,
            linewidths=marker_linewidth,
            label="Chosen thresholds",
            zorder=10,
        )
        ax3.legend(loc="best", frameon=True)
    except NameError:
        print("WARNING: thresholds not defined; skipping marker.")

fig3.savefig(os.path.join(out_dir, "proportion_induction_heads.pdf"))
fig3.savefig(os.path.join(out_dir, "proportion_induction_heads.svg"))
print(f"Saved to {out_dir}")
plt.show()

In [ ]:
# # Combined 1x3 plot for the last three heatmap-style figures:
# #   (1) P(sharp head | induction head)     vs (copy threshold, entropy threshold (normalized))
# #   (2) P(induction head | sharp head)     vs (copy threshold, entropy threshold (normalized))
# #   (3) Proportion of induction heads      vs (copy threshold, prefix threshold)

# # Requires these from the previous cell:
# #   p_sharp_given_copy_prefix, p_copy_prefix_given_sharp, copy_score_thresholds, entropy_thresholds_norm, _to_ent_norm

# # ---------------------
# # Contour plot data prep
# # ---------------------
# rel_attr = relative_attributions_mean.detach().cpu()
# right_scores = right_scores_mean.detach().cpu()

# rel_attr_flat = rel_attr.flatten()
# right_flat = right_scores.flatten()

# # =====================
# # Config (edit these)
# # =====================
# n_Tc = 100  # x-resolution (copy threshold)
# n_Tr = 100  # y-resolution (prefix threshold)

# # If True, pick ranges from data quantiles; if False, use the manual ranges below.
# use_quantile_bounds = False
# Tc_quantiles = (0.01, 0.99)
# Tr_quantiles = (0.01, 0.99)

# # Manual ranges (only used when use_quantile_bounds=False)
# Tc_manual = (1.0, 5.0)
# Tr_manual = (0.25, 0.8)

# # Color scale: choose a robust min/max from the computed probability grid
# use_robust_color_scale = True
# p_color_quantiles = (0.01, 0.99)  # (low, high) quantiles over p_both
# p_color_floor = 0.0  # force vmin >= this (often 0.0)
# n_color_levels = 21  # number of filled contour levels
# add_contour_lines = True

# # Marker for the thresholds used earlier in the notebook
# show_threshold_marker = True
# marker_color = "red"
# marker_size = 140
# marker_linewidth = 2.5

# # =====================
# # Range selection (contour plot)
# # =====================
# rel_np = rel_attr_flat.numpy()
# right_np = right_flat.numpy()

# if use_quantile_bounds:
#     Tc_low, Tc_high = (
#         float(np.quantile(rel_np, Tc_quantiles[0])),
#         float(np.quantile(rel_np, Tc_quantiles[1])),
#     )
#     Tr_low, Tr_high = (
#         float(np.quantile(right_np, Tr_quantiles[0])),
#         float(np.quantile(right_np, Tr_quantiles[1])),
#     )
# else:
#     Tc_low, Tc_high = map(float, Tc_manual)
#     Tr_low, Tr_high = map(float, Tr_manual)

# # Guard against degenerate ranges
# if not (Tc_high > Tc_low):
#     Tc_high = float(np.nextafter(Tc_low, np.inf))
# if not (Tr_high > Tr_low):
#     Tr_high = float(np.nextafter(Tr_low, np.inf))

# copy_score_thresholds_contour = np.linspace(Tc_low, Tc_high, n_Tc)  # x-axis
# prefix_score_thresholds = np.linspace(Tr_low, Tr_high, n_Tr)  # y-axis

# # =====================
# # Compute contour grid
# # =====================
# # Grid: rows=prefix_score_thresholds (y), cols=copy_score_thresholds_contour (x)
# p_both = np.zeros((len(prefix_score_thresholds), len(copy_score_thresholds_contour)), dtype=np.float64)
# num_heads = float(rel_attr_flat.numel())

# for yi, prefix_score_threshold in enumerate(prefix_score_thresholds):
#     for xi, copy_score_threshold in enumerate(copy_score_thresholds_contour):
#         mask = (rel_attr_flat > float(copy_score_threshold)) & (right_flat > float(prefix_score_threshold))
#         p_both[yi, xi] = float(mask.sum().item()) / num_heads

# # =====================
# # Color scaling (data-driven)
# # =====================
# p_flat = p_both.ravel()
# p_min = float(np.nanmin(p_flat))
# p_max = float(np.nanmax(p_flat))

# if use_robust_color_scale:
#     p_lo = float(np.quantile(p_flat, p_color_quantiles[0]))
#     p_hi = float(np.quantile(p_flat, p_color_quantiles[1]))
# else:
#     p_lo, p_hi = p_min, p_max

# p_lo = max(float(p_color_floor), p_lo)
# if not (p_hi > p_lo):
#     p_hi = float(np.nextafter(p_lo, np.inf))

# levels = np.linspace(p_lo, p_hi, n_color_levels)

# # =====================
# # Plot: 1x3 combined grid
# # =====================
# fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
# ax0, ax1, ax2 = axes

# # Heatmaps use the same extent in (copy threshold, entropy threshold (normalized))
# extent_hm = [
#     float(copy_score_thresholds.min()),
#     float(copy_score_thresholds.max()),
#     float(np.min(entropy_thresholds_norm)),
#     float(np.max(entropy_thresholds_norm)),
# ]

# cmap_hm = plt.cm.viridis.copy()
# cmap_hm.set_bad(color="lightgray")

# # (1) P(sharp | induction) — no colorbar
# im0 = ax0.imshow(
#     np.ma.masked_invalid(p_sharp_given_copy_prefix),
#     origin="lower",
#     aspect="auto",
#     extent=extent_hm,
#     vmin=0.0,
#     vmax=1.0,
#     cmap=cmap_hm,
# )
# ax0.set_title("P(sharp head | induction head)")
# ax0.set_xlabel("Copy score threshold")
# ax0.set_ylabel("Entropy threshold (normalized)")

# # (2) P(induction | sharp) — with colorbar
# im1 = ax1.imshow(
#     np.ma.masked_invalid(p_copy_prefix_given_sharp),
#     origin="lower",
#     aspect="auto",
#     extent=extent_hm,
#     vmin=0.0,
#     vmax=1.0,
#     cmap=cmap_hm,
# )
# ax1.set_title("P(induction head | sharp head)")
# ax1.set_xlabel("Copy score threshold")
# ax1.set_ylabel("Entropy threshold (normalized)")
# cbar1 = fig.colorbar(im1, ax=ax1)
# cbar1.set_label("Probability")

# # (3) Proportion of induction heads — with colorbar
# cf = ax2.contourf(
#     copy_score_thresholds_contour,
#     prefix_score_thresholds,
#     p_both,
#     levels=levels,
#     cmap="viridis",
#     extend="max",
# )
# if add_contour_lines:
#     ax2.contour(
#         copy_score_thresholds_contour,
#         prefix_score_thresholds,
#         p_both,
#         levels=levels,
#         colors="k",
#         linewidths=0.3,
#         alpha=0.35,
#     )
# ax2.set_title("Proportion of induction heads")
# ax2.set_xlabel("Copy score threshold")
# ax2.set_ylabel("Prefix score threshold")
# cbar2 = fig.colorbar(cf, ax=ax2)
# cbar2.set_label("Proportion of heads")
# cbar2.ax.set_title(f"[{p_lo:.3f}, {p_hi:.3f}]", fontsize=9)

# # ---------------------
# # Threshold markers
# # ---------------------
# if show_threshold_marker:
#     # For the two probability heatmaps: marker is (copy threshold, entropy threshold (normalized))
#     try:
#         hm_x = float(correct_token_threshold)
#         hm_y = float(_to_ent_norm(float(entropy_threshold)))
#         ax0.scatter(
#             [hm_x],
#             [hm_y],
#             marker="x",
#             c=marker_color,
#             s=marker_size,
#             linewidths=marker_linewidth,
#             zorder=10,
#         )
#         ax1.scatter(
#             [hm_x],
#             [hm_y],
#             marker="x",
#             c=marker_color,
#             s=marker_size,
#             linewidths=marker_linewidth,
#             zorder=10,
#         )

#         # For the contour: marker is (copy threshold, prefix threshold)
#         ct = float(correct_token_threshold)
#         rt = float(right_score_threshold)
#         ax2.scatter(
#             [ct],
#             [rt],
#             marker="x",
#             c=marker_color,
#             s=marker_size,
#             linewidths=marker_linewidth,
#             label="Chosen thresholds",
#             zorder=10,
#         )
#         # Legend ONLY on the rightmost plot
#         ax2.legend(loc="best", frameon=True)
#     except NameError:
#         print("WARNING: thresholds not defined; skipping marker.")

# out_dir = "../../figs/induction_heads/threshold_heatmaps"
# os.makedirs(out_dir, exist_ok=True)
# fig.savefig(os.path.join(out_dir, "induction_head_threshold_heatmaps.pdf"))
# fig.savefig(os.path.join(out_dir, "induction_head_threshold_heatmaps.svg"))

# plt.show()